In [1]:
import yfinance as yf

# Basic Data Retrieval

In [3]:
ticker = "EPAM"
epam = yf.Ticker(ticker)

In [6]:
data = epam.fast_info

In [12]:
print(f"--- {ticker} Stock Monitor ---")
print(f"Current Price: ${data['lastPrice']:.2f}")
print(f"Previous Close: ${data['previousClose']:.2f}")
print(f"Day's Range: ${data['dayLow']:.2f} - ${data['dayHigh']:.2f}")
print(f"Year Range: ${data['yearLow']:.2f} - ${data['yearHigh']:.2f}")
print(f"Volume: {data['lastVolume']:,}")

--- EPAM Stock Monitor ---
Current Price: $141.00
Previous Close: $137.98
Day's Range: $134.01 - $141.08
Year Range: $125.57 - $222.53
Volume: 1,434,888


In [10]:
data

lazy-loading dict with keys = ['currency', 'dayHigh', 'dayLow', 'exchange', 'fiftyDayAverage', 'lastPrice', 'lastVolume', 'marketCap', 'open', 'previousClose', 'quoteType', 'regularMarketPreviousClose', 'shares', 'tenDayAverageVolume', 'threeMonthAverageVolume', 'timezone', 'twoHundredDayAverage', 'yearChange', 'yearHigh', 'yearLow']

In [14]:
# Calculate daily change
price_change = data['lastPrice'] - data['previousClose']
percent_change = (price_change / data['previousClose']) * 100

sign = "+" if price_change >= 0 else ""
print(f"Daily Change: {sign}${price_change:.2f} ({sign}{percent_change:.2f}%)")

Daily Change: +$3.02 (+2.19%)


# Fetching Historical Data for Trend Analysis

In [15]:
# Download historical data for the last 30 days
hist = epam.history(period="30d")

if not hist.empty:
    # Calculate a simple 5-day Moving Average
    hist['MA_5'] = hist['Close'].rolling(window=5).mean()

    latest_row = hist.iloc[-1]

    print("\n--- Recent Trend Analysis ---")
    print(f"Date: {latest_row.name.strftime('%Y-%m-%d')}")
    print(f"Closing Price: ${latest_row['Close']:.2f}")
    print(f"5-Day Moving Avg: ${latest_row['MA_5']:.2f}")

    # Simple logic: Is the price above the 5-day average?
    if latest_row['Close'] > latest_row['MA_5']:
        print("Trend: Bullish (Price > 5-Day MA)")
    else:
        print("Trend: Bearish (Price < 5-Day MA)")
else:
    print("No historical data found.")


--- Recent Trend Analysis ---
Date: 2026-02-27
Closing Price: $141.00
5-Day Moving Avg: $133.90
Trend: Bullish (Price > 5-Day MA)


In [17]:
import time
from datetime import datetime

def monitor_stock(symbol, interval_seconds=60):
    ticker = yf.Ticker(symbol)

    print(f"Starting monitor for {symbol} (Updating every {interval_seconds}s)...")
    print("Press Ctrl+C to stop.\n")

    last_price = None

    try:
        while True:
            # Fetch fresh data
            # Using fast_info again for efficiency in a loop
            info = ticker.fast_info

            current_price = info['lastPrice']
            timestamp = datetime.now().strftime("%H:%M:%S")

            # Only print if the price has changed to reduce noise
            if current_price != last_price:
                change = current_price - (info['previousClose'] if 'previousClose' in info else current_price)
                direction = "▲" if change >= 0 else "▼"

                print(f"[{timestamp}] {symbol}: ${current_price:.2f} {direction} ({change:+.2f})")
                last_price = current_price
            else:
                # Optional: Print a heartbeat dot if no change
                print(".", end="", flush=True)

            time.sleep(interval_seconds)

    except KeyboardInterrupt:
        print("\n\nMonitoring stopped by user.")

# Run the monitor for EPAM every 30 seconds
# monitor_stock("EPAM", interval_seconds=30)

In [18]:
print(epam.info.get('marketState')) # Returns 'REGULAR', 'PRE', 'POST', or 'CLOSED'

POST
